# Finetuning Using Google Gemma's Model


In [ ]:
!pip install -q -U bitsandbytes==0.42.0
!pip3 install -q -U peft==0.8.2
!pip install -q -U trl==0.7.10
!pip3 install -q -U accelerate==0.27.1
!pip install -q -U datasets==2.17.0
!pip3 install -q -U transformers==4.38.0



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.9/150.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.7/279.7 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 18.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2023.10.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 116.2 MB/s eta 0:00:00
   ━━━

In [ ]:
import peft
import trl
import transformers

print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("transformers:", transformers.__version__)

peft: 0.19.1
trl: 1.5.1
transformers: 5.11.0


In [ ]:
!pip show peft

Name: peft
Version: 0.8.2
Summary: Parameter-Efficient Fine-Tuning (PEFT)
Home-page: https://github.com/huggingface/peft
Author: The HuggingFace team
Author-email: sourab@huggingface.co
License: Apache
Location: /usr/local/lib/python3.12/dist-packages
Requires: accelerate, huggingface-hub, numpy, packaging, psutil, pyyaml, safetensors, torch, tqdm, transformers
Required-by: 


In [ ]:
!pip uninstall -y peft trl transformers accelerate diffusers

In [ ]:
!pip install -U peft
!pip install -U trl
!pip install -U transformers
!pip install -U accelerate

  Using cached trl-1.5.1-py3-none-any.whl.metadata (11 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached transformers-5.11.0-py3-none-any.whl.metadata (33 kB)
  Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached huggingface_hub-1.19.0-py3-none-any.whl.metadata (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import transformers
import torch
from google.colab import userdata
from datasets import load_dataset
from trl import SFTTrainer
from peft import LoraConfig
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig, GemmaTokenizer



In [ ]:
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')


### Prerequisites
* nf4(4-bit NormalFloat(NF4)) : https://www.kaggle.com/code/lorentzyeung/what-s-4-bit-quantization-how-does-it-help-llama2


In [ ]:
model_id = "google/gemma-2b"
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

okenizer = AutoTokenizer.from_pretrained(model_id, token=os.environ['HF_TOKEN'])
model = AutoModelForCausalLM.from_pretrained(model_id,
                                             quantization_config=bnb_config,
                                             device_map={"":0},
                                             token=os.environ['HF_TOKEN'])
HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/google/gemma-2b/resolve/main/config.json'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

GatedRepoError                            Traceback (most recent call last)
GatedRepoError: 403 Client Error. (Request ID: Root=1-6a2a9530-4daadf773aa9afab62aa3c25;f8512f8a-8fff-41e0-bba3-89b07d451510)

Cannot access gated repo for url https://huggingface.co/google/gemma-2b/resolve/main/config.json.
Access to model google/gemma-2b is restricted and you are not in the authorized list. Visit https://huggingface.co/google/gemma-2b to ask for access.

The above exception was the direct cause of the following exception:

OSError                                   Traceback (most recent call last)
OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-2b.
403 Client Error. (Request ID: Root=1-6a2a9530-4daadf773aa9afab62aa3c25;f8512f8a-8fff-41e0-bba3-89b07d451510)

Cannot access gated repo for url https://huggingface.co/google/gemma-2b/resolve/main/config.json.
Access to model google/gemma-2b is restricted and you are not in the authorized list. Visit https://huggingface.co/google/gemma-2b to ask for access.

During handling of the above exception, another exception occurred:

HTTPStatusError                           Traceback (most recent call last)
HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/google/gemma-2b/resolve/main/config.json'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

GatedRepoError                            Traceback (most recent call last)
GatedRepoError: 403 Client Error. (Request ID: Root=1-6a2a9532-615058aa08232ca271c0506c;07e62933-3085-4039-944a-992793c309a0)

Cannot access gated repo for url https://huggingface.co/google/gemma-2b/resolve/main/config.json.
Access to model google/gemma-2b is restricted and you are not in the authorized list. Visit https://huggingface.co/google/gemma-2b to ask for access.

The above exception was the direct cause of the following exception:

OSError                                   Traceback (most recent call last)
/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py in cached_files(path_or_repo_id, filenames, cache_dir, force_download, proxies, token, revision, local_files_only, subfolder, repo_type, user_agent, _raise_exceptions_for_gated_repo, _raise_exceptions_for_missing_entries, _raise_exceptions_for_connection_errors, _commit_hash, tqdm_class, **deprecated_kwargs)
    501             if not _raise_exceptions_for_gated_repo:
    502                 return None
--> 503             raise OSError(
    504                 "You are trying to access a gated repo.\nMake sure to have access to it at "
    505                 f"https://huggingface.co/{path_or_repo_id}.\n{str(e)}"

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-2b.
403 Client Error. (Request ID: Root=1-6a2a9532-615058aa08232ca271c0506c;07e62933-3085-4039-944a-992793c309a0)

Cannot access gated repo for url https://huggingface.co/google/gemma-2b/resolve/main/config.json.
Access to model google/gemma-2b is restricted and you are not in the authorized list. Visit https://huggingface.co/google/gemma-2b to ask for access.

Your code is correct now. The problem is not the token code. The error says:

Cannot access gated repo for url https://huggingface.co/google/gemma-2b
Access to model google/gemma-2b is restricted and you are not in the authorized list

You created the Hugging Face token, but your Hugging Face account has not accepted access to Gemma.

Do this:

1. Open Gemma model page

Go here:

https://huggingface.co/google/gemma-2b

Make sure you are logged in with the same account (nidhisah88).

You will see something like:

Access Gemma
Agree and access repository

Click:

Agree and access repository

Accept Google's license.

2. Check access

After accepting, open:

https://huggingface.co/settings/gated-repos

You should see:

google/gemma-2b     Access granted
3. Re-login in Colab

Restart runtime:

Runtime → Restart session

Then:

from huggingface_hub import login
from google.colab import userdata
import os

HF_TOKEN = userdata.get("HF_TOKEN")

login(HF_TOKEN)

os.environ["HF_TOKEN"] = HF_TOKEN
4. Test access

Run:

from huggingface_hub import whoami

print(whoami())

You should see your HF username.

Then:

from huggingface_hub import model_info

model_info(
    "google/gemma-2b",
    token=os.environ["HF_TOKEN"]
)

If it works, you will get model details.

5. Load model again
model_id = "google/gemma-2b"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=os.environ["HF_TOKEN"]
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=os.environ["HF_TOKEN"]
)

One more thing: for fine-tuning in 2026, google/gemma-2b is an older Gemma model. You may prefer:

model_id = "google/gemma-2-2b"

or

model_id = "google/gemma-2-9b"

after accepting access on that model page too. The same token will work.

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

True
12.8
Tesla T4


In [ ]:
!pip uninstall -y bitsandbytes
!pip install bitsandbytes==0.46.1

  Using cached bitsandbytes-0.46.1-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.46.1-py3-none-manylinux_2_24_x86_64.whl (72.9 MB)


In [ ]:
import bitsandbytes

print(bitsandbytes.__version__)

0.46.1


In [ ]:
# !pip install -U bitsandbytes

In [ ]:
import bitsandbytes

print(bitsandbytes.__version__)

0.46.1


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id, token=os.environ['HF_TOKEN'])
model = AutoModelForCausalLM.from_pretrained(model_id,
                                             quantization_config=bnb_config,
                                             device_map={"":0},
                                             token=os.environ['HF_TOKEN'])

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/33.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [ ]:
text = "Quote: Imagination is more,"
device = "cuda:0"
inputs = tokenizer(text, return_tensors="pt").to(device)

outputs = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Quote: Imagination is more, than knowledge.

I am a self-taught artist, born in the Philippines. I have been


In [ ]:
os.environ["WANDB_DISABLED"] = "false"

In [ ]:
lora_config = LoraConfig(
    r = 8,
    target_modules = ["q_proj", "o_proj", "k_proj", "v_proj",
                      "gate_proj", "up_proj", "down_proj"],
    task_type = "CAUSAL_LM",
)

In [ ]:
from datasets import load_dataset

data = load_dataset("Abirate/english_quotes")
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)

README.md:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [ ]:
data['train']['quote']

Column(['“Be yourself; everyone else is already taken.”', "“I'm selfish, impatient and a little insecure. I make mistakes, I am out of control and at times hard to handle. But if you can't handle me at my worst, then you sure as hell don't deserve me at my best.”", "“Two things are infinite: the universe and human stupidity; and I'm not sure about the universe.”", '“So many books, so little time.”', '“A room without books is like a body without a soul.”', ...])

In [ ]:
def formatting_func(example):
    text = f"Quote: {example['quote'][0]}\nAuthor: {example['author'][0]}"
    return [text]

In [ ]:
data['train']

Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 2508
})

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=data["train"],
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=2,
        max_steps=100,
        learning_rate=2e-4,
        # fp16=True,
        fp16=False,
        bf16=True,
        logging_steps=1,
        output_dir="outputs",
        optim="paged_adamw_8bit"
    ),
    peft_config=lora_config,
    formatting_func=formatting_func,
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
trainer.train()

Step,Training Loss
1,2.687388
2,1.709502
3,2.612417
4,2.853542
5,2.401432
6,2.518223
7,2.966690
8,2.299125
9,3.266305
10,2.314212


TrainOutput(global_step=100, training_loss=2.127069571018219, metrics={'train_runtime': 251.2837, 'train_samples_per_second': 1.592, 'train_steps_per_second': 0.398, 'total_flos': 189744345784320.0, 'train_loss': 2.127069571018219, 'epoch': 0.1594896331738437})

In [ ]:
text = "Quote: A woman is like a tea bag;"
device = "cuda:0"
inputs = tokenizer(text, return_tensors="pt").to(device)

outputs = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Quote: A woman is like a tea bag; you can't tell how strong she is until you put her in hot water.

I'


In [ ]:
text = "Quote: Outside of a dog, a book is man's"
device = "cuda:0"
inputs = tokenizer(text, return_tensors="pt").to(device)

outputs = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Quote: Outside of a dog, a book is man's best friend. Inside of a dog, it's too dark to read.

-Groucho
